Setup

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from src.preprocessing import load_data, clean_data, split_data, build_preprocessor

data = clean_data(load_data())
X_train, X_test, y_train, y_test = split_data(data)
print(X_train.shape, y_train.mean().round(4))

(120000, 10) 0.0668


Define the models

In [2]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, min_samples_leaf=5,
        class_weight="balanced_subsample", n_jobs=-1, random_state=42,
    ),
    "Gradient Boosting": HistGradientBoostingClassifier(
        class_weight="balanced", random_state=42,
    ),
}

Cross-validate

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision"}

results = []
for name, model in models.items():
    pipe = Pipeline([("prep", build_preprocessor()), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    results.append({
        "model": name,
        "ROC-AUC": scores["test_roc_auc"].mean(),
        "ROC-AUC std": scores["test_roc_auc"].std(),
        "PR-AUC": scores["test_pr_auc"].mean(),
    })
    print(f"{name} done")

results_df = pd.DataFrame(results).round(4).sort_values("ROC-AUC", ascending=False)
results_df

Logistic Regression done
Random Forest done
Gradient Boosting done


,model,ROC-AUC,ROC-AUC std,PR-AUC
2,Gradient Boosting,0.8633,0.0026,0.3937
1,Random Forest,0.8555,0.0024,0.3798
0,Logistic Regression,0.8546,0.0044,0.3806
